# Lab 4: Evaluating search — Precision, Recall, F1, MAP and NDCG
**Course:** Information Retrieval (IR 2026), CADT  |  **Session 4**  |  **Time:** 3 hours  |  *Student version*

In Lab 3, TF-IDF and BM25 **disagreed** on how to rank documents for the query "python" — but which ranking was actually better? Eyeballing a list does not scale and is not evidence. Today you build the measuring tools of IR: Precision, Recall, F1, Average Precision, MAP and NDCG — and use them to settle, with numbers, whether TF-IDF or BM25 wins on your own test collection.

**By the end of this lab you can:**
1. Explain what a test collection (queries + documents + relevance judgments) is, and why graded judgments carry more information than binary ones.
2. Compute Precision, Recall and F1 at a cutoff k.
3. Compute Average Precision for one query and Mean Average Precision (MAP) across queries.
4. Compute NDCG using graded relevance judgments.
5. Explain, with your own numbers, a case where MAP and NDCG disagree about which ranking is better.
6. Judge relevance yourself and discuss disagreement with a partner.

**How to work:** run the cells in order, replace every `# TODO`, and write your answers in the *Answer* cells. Part 6 is a paired activity.

## Part 0. Setup and recap (about 10 min)
We reuse the same 30-document collection and the TF-IDF / BM25 ranking functions from Lab 3. Recall: for the query "python", TF-IDF ranked `d04` first; BM25 ranked `d18` first. Today you decide, with evidence, whether that difference actually matters.

In [14]:
import re, math
import pandas as pd
import matplotlib.pyplot as plt
docs = {
    "d01": "Budget guesthouses in Phnom Penh offer clean rooms near the Riverside for under twenty dollars a night.",
    "d02": "Cheap hotel deals in Siem Reap are easy to find, especially during the rainy season.",
    "d03": "The Cambodia Academy of Digital Technology offers degrees in software engineering, data science and cybersecurity.",
    "d04": "Python is a popular programming language for data science, web development and automation.",
    "d05": "A ball python is a small, docile snake that is often kept as a pet.",
    "d06": "Java is a statically typed language widely used for enterprise applications and Android development.",
    "d07": "Java is an island in Indonesia with more than one hundred million inhabitants.",
    "d08": "Angkor Wat is the largest religious monument in the world and attracts millions of visitors every year.",
    "d09": "The Mekong River flows through six countries and supports fishing and rice farming along its banks.",
    "d10": "Rice is the main crop in Cambodia, and farmers plant it at the start of the rainy season.",
    "d11": "Fried noodles and fish amok are popular dishes in Cambodian restaurants.",
    "d12": "Buses run every hour from Phnom Penh to Siem Reap, and the trip takes about six hours.",
    "d13": "Tuk-tuk drivers in the capital usually negotiate the fare before the ride starts.",
    "d14": "Electric vehicles are becoming more common as battery prices fall.",
    "d15": "An automobile engine converts fuel into motion using internal combustion.",
    "d16": "Her cat sleeps all day, but at night it starts running around the house.",
    "d17": "The library catalog lists every category of book, from fiction to computer science.",
    "d18": "To concatenate two strings in Python, use the plus operator.",
    "d19": "The marathon runners ran through the city streets despite the heat.",
    "d20": "Apartments with 2 bedrooms near the university are popular with students.",
    "d21": "Machine learning models learn patterns from data instead of following hand-written rules.",
    "d22": "Search engines use crawlers to collect web pages and build an index for fast lookup.",
    "d23": "A database stores structured records in tables and answers precise queries written in SQL.",
    "d24": "The government announced new investment in digital skills and internet access for rural areas.",
    "d25": "Coffee shops in the capital are popular places for students to study and meet friends.",
    "d26": "Khmer script has no spaces between words, which makes text processing harder.",
    "d27": "Airlines added more flights to the coastal city of Sihanoukville for the holiday season.",
    "d28": "Learning to program requires regular practice, not only reading books.",
    "d29": "The National Museum in Phnom Penh displays Khmer sculpture from the Angkorian period.",
    "d30": "Affordable accommodation for backpackers is available in Kampot, including hostels and homestays.",
}

STOP_WORDS = set("""a an the is are was were be been being of to in on at for with and or not
this that these those it its as by from into over under about than then so if but""".split())
SUFFIXES = ["ational", "tional", "alize", "ation", "ative", "fulness", "ousness",
            "iveness", "ing", "ies", "ied", "ly", "ed", "es", "s"]

def simple_stem(word):
    for suf in SUFFIXES:
        if word.endswith(suf) and len(word) - len(suf) >= 3:
            return word[:-len(suf)]
    return word

def preprocess(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    tokens = [t for t in tokens if t not in STOP_WORDS]
    return [simple_stem(t) for t in tokens]

doc_tokens = {doc_id: preprocess(text) for doc_id, text in docs.items()}

doc_len = {doc_id: len(toks) for doc_id, toks in doc_tokens.items()}

N = len(docs)

AVGDL = sum(doc_len.values()) / N

def tf(term, doc_id): return doc_tokens[doc_id].count(term)

def df(term): return sum(1 for d in doc_tokens if term in doc_tokens[d])

def idf(term):
    d = df(term)
    return math.log(N / d) if d else 0.0

def tfidf_vector(doc_id):
    return {t: tf(t, doc_id) * idf(t) for t in set(doc_tokens[doc_id])}

def cosine_similarity(v1, v2):
    shared = set(v1) & set(v2)
    dot = sum(v1[t] * v2[t] for t in shared)
    n1 = math.sqrt(sum(x * x for x in v1.values())); n2 = math.sqrt(sum(x * x for x in v2.values()))
    return dot / (n1 * n2) if n1 and n2 else 0.0

def query_vector(query_text):
    terms = preprocess(query_text)
    tf_q = {}
    for t in terms: tf_q[t] = tf_q.get(t, 0) + 1
    return {t: tf_q[t] * idf(t) for t in tf_q}

def rank_tfidf(query_text, top_k=10):
    qv = query_vector(query_text)
    scored = [(d, cosine_similarity(qv, tfidf_vector(d))) for d in docs]
    scored = [s for s in scored if s[1] > 0]
    return sorted(scored, key=lambda x: -x[1])[:top_k]

def idf_bm25(term):
    d = df(term)
    return math.log((N - d + 0.5) / (d + 0.5) + 1)

def bm25_score(query_text, doc_id, k1=1.5, b=0.75):
    terms = set(preprocess(query_text))
    score = 0.0
    dl = doc_len[doc_id]
    for t in terms:
        f = tf(t, doc_id)
        if f == 0: continue
        score += idf_bm25(t) * (f * (k1 + 1)) / (f + k1 * (1 - b + b * dl / AVGDL))
    return score

def rank_bm25(query_text, top_k=10, k1=1.5, b=0.75):
    scored = [(d, bm25_score(query_text, d, k1, b)) for d in docs]
    scored = [s for s in scored if s[1] > 0]
    return sorted(scored, key=lambda x: -x[1])[:top_k]

print("Pipeline ready:", N, "documents, avg length", round(AVGDL, 1))

Pipeline ready: 30 documents, avg length 9.5


## Part 1. Building a test collection (about 20 min)
A **test collection** has three parts: documents (our 30), queries (below, from Lab 1), and **relevance judgments** (qrels) — which documents are relevant to each query, decided by a human.

**Binary** judgments say yes/no. **Graded** judgments say HOW relevant, e.g. on a 0–3 scale (0 = not relevant, 3 = highly relevant). Graded judgments carry more information: they let us reward a ranking for putting the BEST document first, not just any relevant document.

In [15]:
needs = {
    "N1": {"need": 'Find a cheap place to stay in Cambodia', "queries": ['cheap hotel', 'budget accommodation']},
    "N2": {"need": 'Learn about the Python programming language', "queries": ['python']},
    "N3": {"need": 'Travel from Phnom Penh to Siem Reap', "queries": ['travel Phnom Penh Siem Reap']},
    "N4": {"need": 'Information about cars', "queries": ['car']},
    "N5": {"need": 'Documents about cats', "queries": ['cat']},
    "N6": {"need": 'Text about running', "queries": ['running']},
}

# Binary judgments (same as Labs 1-3; N6 excludes d12, a borderline case about bus schedules, not running)
binary_judgments = {
    "N1": {"d01", "d02", "d30"}, "N2": {"d04", "d18"}, "N3": {"d12"},
    "N4": {"d14", "d15"}, "N5": {"d16"}, "N6": {"d16", "d19"},
}

# Graded judgments (0-3): the SAME documents, but now how good a match each one is
graded_judgments = {
    "N1": {"d02": 3, "d01": 2, "d30": 2},
    "N2": {"d04": 3, "d18": 2, "d05": 0},
    "N3": {"d12": 3, "d02": 1, "d29": 1, "d01": 1},
    "N4": {"d15": 3, "d14": 2},
    "N5": {"d16": 3},
    "N6": {"d19": 3, "d16": 2, "d12": 0},
}
pd.DataFrame([{"need": k, **v} for k, v in graded_judgments.items()]).set_index("need")

,d02,d01,d30,d04,d18,d05,d12,d29,d15,d14,d16,d19
need,,,,,,,,,,,,
N1,3.0,2.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
N2,NaN,NaN,NaN,3.0,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
N3,1.0,1.0,NaN,NaN,NaN,NaN,3.0,1.0,NaN,NaN,NaN,NaN
N4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,2.0,NaN,NaN
N5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.0,NaN
N6,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,2.0,3.0


**Answer 1.1** Look at N2 ("python"): `d04` is graded 3 and `d18` is graded 2, but both were counted as simply "relevant" in the binary judgments. What real difference between these two documents does the graded score capture that binary judgment misses?

*Your answer:*

Binary judgments mark both simply as relevant  is yes, so they miss that d04 is more useful and better matches the query than d18.

## Part 2. Precision, Recall and F1 at k (about 25 min)
Recap from Session 1, now formalized at a cutoff **k** (the number of results actually looked at):

$$P@k = \frac{\text{relevant documents in the top } k}{k} \qquad R@k = \frac{\text{relevant documents in the top } k}{\text{total relevant documents}}$$

In [16]:
def precision_at_k(ranked_ids, relevant, k):
    """Fraction of the top k results that are relevant. ranked_ids: list of doc ids, best first."""
    # TODO: look at ranked_ids[:k]; count how many are in `relevant`; divide by k
    relevant_count = sum(doc_id in relevant for doc_id in ranked_ids[:k])

    return relevant_count / k if k > 0 else 0.0

def recall_at_k(ranked_ids, relevant, k):
    """Fraction of ALL relevant documents that appear in the top k."""
    # TODO: same count as above; divide by len(relevant) instead (handle relevant being empty)
    if not relevant:
        return 0.0

    relevant_count = sum(doc_id in relevant for doc_id in ranked_ids[:k])
    return relevant_count / len(relevant)


def f1_score(precision, recall):
    """Harmonic mean of precision and recall. Return 0.0 if both are 0."""
    # TODO
    if precision + recall == 0 :
        return 0.0
    return 2*(precision*recall)/(precision+recall)

In [17]:
rows = []
for nid, info in needs.items():
    q = info["queries"][0]
    relevant = binary_judgments[nid]
    for model, rank_fn in [("TF-IDF", rank_tfidf), ("BM25", rank_bm25)]:
        ranked = [d for d, _ in rank_fn(q, top_k=10)]
        p, r = precision_at_k(ranked, relevant, 3), recall_at_k(ranked, relevant, 3)
        rows.append({"need": nid, "query": q, "model": model, "P@3": round(p, 2),
                     "R@3": round(r, 2), "F1@3": round(f1_score(p, r), 2)})
pd.DataFrame(rows)

,need,query,model,P@3,R@3,F1@3
0,N1,cheap hotel,TF-IDF,0.33,0.33,0.33
1,N1,cheap hotel,BM25,0.33,0.33,0.33
2,N2,python,TF-IDF,0.67,1.00,0.80
3,N2,python,BM25,0.67,1.00,0.80
4,N3,travel Phnom Penh Siem Reap,TF-IDF,0.33,1.00,0.50
5,N3,travel Phnom Penh Siem Reap,BM25,0.33,1.00,0.50
6,N4,car,TF-IDF,0.00,0.00,0.00
7,N4,car,BM25,0.00,0.00,0.00
8,N5,cat,TF-IDF,0.33,1.00,0.50
9,N5,cat,BM25,0.33,1.00,0.50


**Answer 2.1** Find a need where P@3 and R@3 are very different from each other. What does that tell you about the ranking at that cutoff?

*Your answer:*

For N3, only 1 of the top 3 documents is relevant. They retrieve all relevant documents but also include 2 irrelevant ones.

## Part 3. Average Precision and MAP (about 30 min)
P@k and R@k need you to pick a k. **Average Precision (AP)** avoids that choice: it looks at precision at EVERY point a relevant document appears, then averages those precision values. **MAP** (Mean Average Precision) is simply the mean of AP across all your queries — a single number summarizing a whole system.

$$AP = \frac{1}{|\text{relevant}|}\sum_{\text{ranks } i \text{ of relevant docs}} \frac{\text{relevant docs in top } i}{i}$$

In [18]:
def average_precision(ranked_ids, relevant):
    """Average Precision for one query. 0.0 if `relevant` is empty."""
    # TODO: walk ranked_ids; keep a running count of hits; each time you hit a relevant
    #       document, add (hits so far / current rank) to a running sum; divide by len(relevant)
    if not relevant:
        return 0.0

    hits = 0
    total = 0
    rank = 0

    for doc in ranked_ids:
        rank += 1

        if doc in relevant:
            hits += 1
            total += hits / rank

    return total / len(relevant)


def mean_average_precision(ap_scores):
    """MAP: the mean of a list of AP values."""
    # TODO
    return sum(ap_scores) / len(ap_scores) if ap_scores else 0.0

In [19]:
ap_rows, aps_tfidf, aps_bm25 = [], [], []
for nid, info in needs.items():
    q = info["queries"][0]
    relevant = binary_judgments[nid]
    rt = [d for d, _ in rank_tfidf(q, top_k=10)]
    rb = [d for d, _ in rank_bm25(q, top_k=10)]
    apt, apb = average_precision(rt, relevant), average_precision(rb, relevant)
    aps_tfidf.append(apt); aps_bm25.append(apb)
    ap_rows.append({"need": nid, "query": q, "AP (TF-IDF)": round(apt, 3), "AP (BM25)": round(apb, 3)})

display(pd.DataFrame(ap_rows))
print(f"\nMAP (TF-IDF) = {mean_average_precision(aps_tfidf):.3f}")
print(f"MAP (BM25)   = {mean_average_precision(aps_bm25):.3f}")

,need,query,AP (TF-IDF),AP (BM25)
0,N1,cheap hotel,0.333,0.333
1,N2,python,1.000,0.833
2,N3,travel Phnom Penh Siem Reap,1.000,1.000
3,N4,car,0.000,0.000
4,N5,cat,1.000,1.000
5,N6,running,0.500,0.500



MAP (TF-IDF) = 0.639
MAP (BM25)   = 0.611


**Answer 3.1** Which model has the higher MAP on this query set? Find the ONE query that contributes most to the difference, and explain why its AP values differ.

*Your answer:*
TF-IDF has higher MAP than BM25. The main difference is query N2 python. BM25 ranks d05 before d04 because d05 is shorter. TF-IDF uses cosine similarity and ranks d05 last because its rare words  reduce the similarity score. 

In [20]:
print("TF-IDF:", rank_tfidf("python"))
print("BM25:", rank_bm25("python"))

TF-IDF: [('d04', 0.2853757796047736), ('d18', 0.2742802877814755), ('d05', 0.24789249585861994)]
BM25: [('d18', 2.4709520662987514), ('d05', 2.3446922143114945), ('d04', 2.2307082571879615)]


## Break (10 min)

## Part 4. NDCG (about 30 min)
AP/MAP only use binary relevance: a document is either "in" or "out". **NDCG** (Normalized Discounted Cumulative Gain) uses GRADED relevance, and — crucially — discounts documents that appear lower in the ranking, so getting the BEST document to rank #1 matters, not just getting A relevant document somewhere near the top.

$$DCG@k = \sum_{i=1}^{k} \frac{2^{\text{rel}_i} - 1}{\log_2(i + 1)} \qquad NDCG@k = \frac{DCG@k}{IDCG@k}$$

`IDCG@k` is the DCG of the IDEAL ranking (documents sorted by true relevance, best first) — it is the highest DCG achievable, so NDCG is always between 0 and 1.

In [21]:
def dcg(relevances):
    """DCG for a list of relevance grades, already in RANKED order (best-guess first)."""
    # TODO: sum (2**rel - 1) / log2(i + 2) for i, rel in enumerate(relevances) -- i starts at 0, so rank = i+1, and i+2 = rank+1
    total = 0
    rank = 1

    for rel in relevances:
        total += (2**rel - 1) / math.log2(rank + 1)
        rank += 1

    return total

def ndcg(ranked_ids, rel_map, k=10):
    """NDCG@k. rel_map: {doc_id: grade}; documents not in rel_map count as grade 0."""
    # TODO: build `relevances` for ranked_ids[:k] using rel_map.get(doc_id, 0);
    #       build `ideal` by sorting rel_map.values() descending, truncated to k;
    #       return dcg(relevances) / dcg(ideal), or 0.0 if dcg(ideal) is 0
    relevances = [rel_map.get(doc, 0) for doc in ranked_ids[:k]]
    ideal = sorted(rel_map.values(), reverse=True)[:k]

    ideal_score = dcg(ideal)

    if ideal_score == 0:
        return 0.0

    return dcg(relevances) / ideal_score

In [22]:
ndcg_rows, ndcgs_tfidf, ndcgs_bm25 = [], [], []
for nid, info in needs.items():
    q = info["queries"][0]
    rel_map = graded_judgments[nid]
    rt = [d for d, _ in rank_tfidf(q, top_k=10)]
    rb = [d for d, _ in rank_bm25(q, top_k=10)]
    nt, nb = ndcg(rt, rel_map), ndcg(rb, rel_map)
    ndcgs_tfidf.append(nt); ndcgs_bm25.append(nb)
    ndcg_rows.append({"need": nid, "query": q, "NDCG (TF-IDF)": round(nt, 3), "NDCG (BM25)": round(nb, 3)})

display(pd.DataFrame(ndcg_rows))
print(f"\nAverage NDCG (TF-IDF) = {sum(ndcgs_tfidf)/len(ndcgs_tfidf):.3f}")
print(f"Average NDCG (BM25)   = {sum(ndcgs_bm25)/len(ndcgs_bm25):.3f}")

,need,query,NDCG (TF-IDF),NDCG (BM25)
0,N1,cheap hotel,0.674,0.674
1,N2,python,1.000,0.731
2,N3,travel Phnom Penh Siem Reap,1.000,1.000
3,N4,car,0.000,0.000
4,N5,cat,1.000,1.000
5,N6,running,0.337,0.337



Average NDCG (TF-IDF) = 0.668
Average NDCG (BM25)   = 0.624


MAP (TF-IDF) = 0.639
MAP (BM25)   = 0.611

**Answer 4.1** Compare your MAP and average-NDCG results. Do they agree on which model is better overall? Does the SIZE of the gap look the same under both metrics?

*Your answer:*
Both metrics show that TF-IDF performs slightly better on this query set, but the difference is small. The gap is larger in NDCG because it considers relevance grades, while MAP uses binary relevance. Neither model can be considered generally better based on these results.

## Part 5. When metrics disagree (about 20 min)
Here is a sharper test. BM25's `b` parameter controls document-length normalization (Lab 3). Vary it and watch AP and NDCG for the "python" query — they do not always move together.

In [23]:
# TODO: for b in [0, 0.75, 1.0], compute rank_bm25("python", b=b), then print
#       its AP (using binary_judgments["N2"]) and its NDCG (using graded_judgments["N2"])
for b in [0, 0.75, 1.0]:
    ranked = [d for d, _ in rank_bm25("python", top_k=10, b=b)]
    ap = average_precision(ranked, binary_judgments["N2"])
    ndcg_score = ndcg(ranked, graded_judgments["N2"])

    print(f"b={b}: AP={ap:.3f}, NDCG={ndcg_score:.3f}")

b=0: AP=0.833, NDCG=0.956
b=0.75: AP=0.833, NDCG=0.731
b=1.0: AP=0.833, NDCG=0.731


**Answer 5.1** As `b` changes, does AP change? Does NDCG change? If they behave differently, explain why — think about what each metric actually looks at (binary "is it relevant" vs. graded "how good is it, and how early does it appear").

*Your answer:*

AP stays the same as b changes, but NDCG changes. This is because AP only considers whether documents are relevant, while NDCG considers relevance grades and ranking positions.

**Answer 5.2** If you were building a system where getting the SINGLE best document to the top mattered most (e.g. a “show me the official page” query), which metric would you trust more to guide your tuning — MAP or NDCG? Why?

*Your answer:*

I would choose NDCG because it gives more importance to highly relevant documents appearing at the top, which is useful when finding the single best document.

## Part 6. Judge it yourself (about 25 min, with a partner)
A new query, not used in any earlier lab: **"Find something about studying or courses in Cambodia."**

Candidate documents: `d03`, `d20`, `d25`, `d28`.

In [24]:
for d in ["d03", "d20", "d25", "d28"]:
    print(d, "->", docs[d])

d03 -> The Cambodia Academy of Digital Technology offers degrees in software engineering, data science and cybersecurity.
d20 -> Apartments with 2 bedrooms near the university are popular with students.
d25 -> Coffee shops in the capital are popular places for students to study and meet friends.
d28 -> Learning to program requires regular practice, not only reading books.


1. **Alone first**, grade each document 0–3 for this need (0 = not relevant, 3 = highly relevant).
2. Compare with a partner. Note every disagreement.
3. Fill in the cell below with the judgments you and your partner AGREED on (or your own, if working alone), then run it.

In [25]:
# TODO: fill in your (agreed) grades, e.g. {"d03": 3, "d20": 1, "d25": 1, "d28": 2}
my_judgments = {
    "d03": 3,
    "d20": 0,
    "d25": 1,
    "d28": 2,
}

**Answer 6.1** Where did you and your partner disagree, and by how much (e.g. one of you said 3, the other said 1)? What made that document ambiguous?

*Your answer:*
d25 were ambiguous because they are related to studying but do not directly answer to query.

**Answer 6.2** Real test collections (like TREC's) are built by professional annotators, and even they disagree sometimes. Why does annotator disagreement matter for how much you should trust a small difference in MAP or NDCG between two systems?

*Your answer:*

small differences in MAP or NDCG may not mean one system is better.

## Part 7. Reflection (about 5 min in class; finish as homework)
Write 4–6 sentences: *Before today, how would you have decided whether TF-IDF or BM25 is "better"? What did Precision/Recall, MAP and NDCG each add that eyeballing a ranked list could not? Use at least one number from Part 3, 4 or 5 as evidence.*

*Your reflection:*

I think TF-IDF better than BM25 for small document collections. Evaluation metrics help us understand how well each model retrieves relevant information. These metrics provide a more reliable and consistent way to compare performance than just looking at ranked results. In Part 3, TF-IDF achieved a higher MAP than BM25.

---
### Submission checklist
- [ ] All `# TODO` cells completed and the notebook runs from top to bottom without errors
- [ ] All *Answer* cells written in your own words
- [ ] Part 3 (MAP) and Part 4 (NDCG) tables completed
- [ ] Part 6 peer judgments recorded, with disagreements noted
- [ ] Notebook saved as `Lab4_<StudentID>.ipynb` and submitted as instructed